# Wiki recommender

## 0. Introduction

## 1. Setup

In [13]:
import pandas as pd
import requests
import bs4
from time import sleep
import re
import numpy as np
import time

## 2. Data Collection (Crawling & Scraping)

In [14]:
class WikiScraper():
    
    def __init__(self, start_url_list, delay=3):
        self.headers = {'User-Agent': 'Recommender'}
        self.delay = delay # in seconds
        self.list_of_articles = []
        self.start_url_list = start_url_list
        
    def _fetch_article(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
        except requests.RequestException:
            return None
        if response.status_code != 200: return None

        parsed = bs4.BeautifulSoup(response.text, "html.parser")

        article = {"url": response.url}
        article["title"] = parsed.select_one("#firstHeading").get_text()
        article["content"] = " ".join(p.get_text() for p in parsed.select("#mw-content-text p"))
        links = self._extract_links(parsed)

        return article, links
        
    def _extract_links(self, parsed):
        links = parsed.find_all('a', attrs={'href': re.compile(r'^https://en\.wikipedia\.org/wiki/')})
        np.random.shuffle(links)
        return [link['href'] for link in links if ':' not in link['href'].split('/wiki/')[1]]
    
    def crawl(self, max_num_articles=50):
        stack = list(reversed(self.start_url_list))
        visited = set()
        
        while stack and len(self.list_of_articles)<max_num_articles:
            url = stack.pop()
            if url in visited:
                continue
            
            visited.add(url)
            
            result = self._fetch_article(url)
            time.sleep(self.delay)
            
            if result is None:
                continue
            
            article, links = result
            stack.extend(links)
            
            if len(article["content"].split()) < 200:
                continue

            self.list_of_articles.append(article)
            
        print(f"Downloaded {len(self.list_of_articles)} articles, visited {len(visited)} URLs")
            
    def get_articles(self):
        return self.list_of_articles              

In [15]:
scraper = WikiScraper(
    ["https://en.wikipedia.org/wiki/Machine_learning",
     "https://en.wikipedia.org/wiki/Jazz"],
    delay=0.5,
)
scraper.crawl(max_num_articles=40)

df = pd.DataFrame(scraper.get_articles())
df["n_words"] = df["content"].str.split().str.len()

print(df[["title", "n_words"]])
print(df["content"].iloc[0][:500])

Downloaded 40 articles, visited 44 URLs
                                      title  n_words
0                          Machine learning     8858
1                 Digital object identifier     2799
2                                      ISBN     3418
3                  Romanization of Armenian     1219
4                                   ISO 639     1226
5                                    ANSI C      823
6                                 IEC 60320     2704
7                 Stage lighting instrument     4524
8                        Luminous intensity      709
9                      Dimensional analysis     8825
10                                  Voltage     1944
11                             Four-current      373
12  Maxwell's equations in curved spacetime     5165
13                            Metric tensor     4542
14              Cartesian coordinate system     4901
15        Parabolic cylindrical coordinates      445
16                   Orthogonal coordinates     2834
17    

## Stemming, lemmatization

In [16]:
import nltk

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords


from nltk.tokenize import word_tokenize, wordpunct_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
_ =nltk.download('stopwords', quiet=True)

In [17]:
wordnet = WordNetLemmatizer()
porter = PorterStemmer()
stop = set(stopwords.words("english"))


In [18]:
def custom_stemmer(text):
    stems = []
    for x in word_tokenize(text.lower()):
        if x.isalpha() and x not in stop:
            stems.append(porter.stem(x))
    return stems

def custom_lemmatizer(text):
    lemms = []
    for x in word_tokenize(text.lower()):
        if x.isalpha() and x not in stop:
            lemms.append(wordnet.lemmatize(x))
    return lemms

In [19]:
df["tokens_stem"] = df["content"].apply(custom_stemmer)
df["tokens_lemma"] = df["content"].apply(custom_lemmatizer)

In [20]:
print(df["tokens_stem"].iloc[0][:20])
print(df["tokens_lemma"].iloc[0][:20])

['machin', 'learn', 'ml', 'field', 'studi', 'artifici', 'intellig', 'concern', 'develop', 'studi', 'statist', 'algorithm', 'learn', 'data', 'gener', 'unseen', 'data', 'thu', 'perform', 'task']
['machine', 'learning', 'ml', 'field', 'study', 'artificial', 'intelligence', 'concerned', 'development', 'study', 'statistical', 'algorithm', 'learn', 'data', 'generalize', 'unseen', 'data', 'thus', 'perform', 'task']


Vocabulary sizes

In [21]:
print(len(set(t for ts in df["tokens_stem"] for t in ts)),
      len(set(t for ts in df["tokens_lemma"] for t in ts))) 

5602 7434


In [22]:
df.to_parquet("articles.parquet", index=False)